In [ ]:
!pip install flash-attn --no-build-isolation

In [ ]:
#!/usr/bin/env python3
"""
Optimized launcher script for full precision LoRA fine-tuning on single H100 GPU.
Fixed for MoE (Mixture of Experts) models like gpt-oss-20b.
"""

import os
import sys

print("="*80)
print("H100 OPTIMIZED FULL PRECISION LORA MODE - MOE COMPATIBLE")
print("Training 20b MoE model with memory optimizations")
print("="*80)
print()

import json
import torch
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional, Dict, List
from datasets import load_dataset, Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq,
    TrainerCallback,
)

import transformers
# Enable Transformers verbose logging
transformers.logging.set_verbosity_info()

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)
import numpy as np

# ============================================================================
# Configuration
# ============================================================================

@dataclass
class Config:
    # Model settings
    model_name: str = "/kaggle/input/models/danielhanchen/gpt-oss-20b/transformers/default/1"
    
    # LoRA settings - Memory optimized for H100 with MoE
    lora_r: int = 32
    lora_alpha: int = 64
    lora_dropout: float = 0.05
    # For MoE models, be careful with expert layers
    lora_target_modules: List[str] = field(default_factory=lambda: [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        # Avoid MLP/expert layers initially for MoE models
    ])
    lora_bias: str = "none"
    
    # Data settings
    dataset_file: str = "NuminaMath-TIR.csv"
    val_split_ratio: float = 0.005
    random_seed: int = 42
    max_seq_length: int = 2048  # Conservative starting point
    
    # Training settings - Optimized for 20b MoE on single H100
    output_dir: str = "./outputs"
    num_train_epochs: int = 2
    per_device_train_batch_size: int = 1
    per_device_eval_batch_size: int = 1
    gradient_accumulation_steps: int = 16
    learning_rate: float = 2e-4
    warmup_ratio: float = 0.03
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    
    # Logging and evaluation
    logging_steps: int = 10
    eval_steps: int = 250
    save_steps: int = 250
    save_total_limit: int = 2
    
    # Optimization settings - H100 optimized
    optim: str = "adamw_torch_fused"
    adam_beta1: float = 0.9
    adam_beta2: float = 0.95
    adam_epsilon: float = 1e-8
    gradient_checkpointing: bool = True
    gradient_checkpointing_use_reentrant: bool = False
    bf16: bool = True
    fp16: bool = False
    tf32: bool = True
    
    # Memory management - CRITICAL for MoE models
    use_cpu_offload: bool = False  # Disable offloading for MoE
    
    # Resume settings
    resume_from_checkpoint: Optional[str] = None
    
    # DeepSpeed settings (optional)
    deepspeed: Optional[str] = None

# ============================================================================
# Memory Monitoring
# ============================================================================

class MemoryMonitorCallback(TrainerCallback):
    """Monitor GPU memory usage during training."""
    
    def __init__(self):
        self.peak_memory = 0
    
    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step % 10 == 0:
            if torch.cuda.is_available():
                allocated = torch.cuda.memory_allocated(0) / 1024**3
                reserved = torch.cuda.memory_reserved(0) / 1024**3
                max_allocated = torch.cuda.max_memory_allocated(0) / 1024**3
                
                self.peak_memory = max(self.peak_memory, allocated)
                
                print(f"\n[Memory] Step {state.global_step}: "
                      f"Allocated: {allocated:.2f}GB, "
                      f"Reserved: {reserved:.2f}GB, "
                      f"Peak: {max_allocated:.2f}GB")
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        """Clear cache after logging to prevent memory buildup."""
        if logs and torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    def on_train_end(self, args, state, control, **kwargs):
        print(f"\n{'='*80}")
        print(f"Peak GPU Memory Usage: {self.peak_memory:.2f}GB")
        print(f"{'='*80}\n")

# ============================================================================
# Data Processing
# ============================================================================

def format_instruction(sample: Dict) -> str:
    """Format the sample into instruction format."""
    prompt = sample["prompt"]
    response = sample["response"]
    
    formatted = f"""<|im_start|>user
{prompt}<|im_end|>
<|im_start|>assistant
{response}<|im_end|>"""
    
    return formatted

def preprocess_function(examples: Dict, tokenizer: AutoTokenizer, max_length: int) -> Dict:
    """Tokenize and prepare the dataset."""
    formatted_texts = [format_instruction(
        {"prompt": p, "response": r}
    ) for p, r in zip(examples["prompt"], examples["response"])]
    
    tokenized = tokenizer(
        formatted_texts,
        max_length=max_length,
        truncation=True,
        padding=False,
        return_tensors=None,
    )
    
    tokenized["labels"] = tokenized["input_ids"].copy()
    
    return tokenized

def load_and_prepare_data(config: Config, tokenizer: AutoTokenizer):
    """Load and prepare train and validation datasets from a CSV file."""
    print(f"Loading dataset from {config.dataset_file}...")

    try:
        df = pd.read_csv(config.dataset_file)
        print(f"Successfully loaded {len(df)} examples from CSV file")
    except Exception as e:
        raise ValueError(f"Error loading CSV file: {e}")
    
    print(f"Available columns: {df.columns.tolist()}")

    # Validate and rename columns
    column_mapping = {}
    
    # Check for prompt/input columns
    if 'problem' in df.columns:
        column_mapping['problem'] = 'prompt'
    elif 'question' in df.columns:
        column_mapping['question'] = 'prompt'
    elif 'input' in df.columns:
        column_mapping['input'] = 'prompt'
    elif 'prompt' in df.columns:
        column_mapping['prompt'] = 'prompt'
    else:
        raise ValueError(
            f"Dataset must contain one of: 'problem', 'question', 'input', or 'prompt' columns. "
            f"Found: {df.columns.tolist()}"
        )
    
    # Check for response/output columns
    if 'solution' in df.columns:
        column_mapping['solution'] = 'response'
    elif 'answer' in df.columns:
        column_mapping['answer'] = 'response'
    elif 'output' in df.columns:
        column_mapping['output'] = 'response'
    elif 'response' in df.columns:
        column_mapping['response'] = 'response'
    else:
        raise ValueError(
            f"Dataset must contain one of: 'solution', 'answer', 'output', or 'response' columns. "
            f"Found: {df.columns.tolist()}"
        )

    # Select and rename columns
    original_columns = list(column_mapping.keys())
    df = df[original_columns].rename(columns=column_mapping)
    
    # Ensure data types are strings and handle missing values
    df['prompt'] = df['prompt'].fillna('').astype(str)
    df['response'] = df['response'].fillna('').astype(str)
    
    # Remove empty rows
    initial_len = len(df)
    df = df[(df['prompt'].str.strip() != '') & (df['response'].str.strip() != '')]
    final_len = len(df)
    
    if final_len < initial_len:
        print(f"Removed {initial_len - final_len} empty rows. Final dataset size: {final_len}")

    if len(df) == 0:
        raise ValueError("No valid data found after cleaning. Please check your CSV file.")

    full_dataset = Dataset.from_pandas(df, preserve_index=False)
    print(f"Total dataset size: {len(full_dataset)}")
    
    split_dataset = full_dataset.train_test_split(
        test_size=config.val_split_ratio,
        seed=config.random_seed,
        shuffle=True
    )
    
    train_dataset = split_dataset["train"]
    val_dataset = split_dataset["test"]
    
    print(f"Train dataset size: {len(train_dataset)} ({(1-config.val_split_ratio)*100:.1f}%)")
    print(f"Validation dataset size: {len(val_dataset)} ({config.val_split_ratio*100:.1f}%)")
    
    train_dataset = train_dataset.map(
        lambda x: preprocess_function(x, tokenizer, config.max_seq_length),
        batched=True,
        remove_columns=train_dataset.column_names,
        desc="Tokenizing train dataset",
    )
    
    val_dataset = val_dataset.map(
        lambda x: preprocess_function(x, tokenizer, config.max_seq_length),
        batched=True,
        remove_columns=val_dataset.column_names,
        desc="Tokenizing validation dataset",
    )
    
    return train_dataset, val_dataset

# ============================================================================
# Model Setup with LoRA (Full Precision - MoE Compatible)
# ============================================================================

def setup_model_and_tokenizer(config: Config):
    """Load and prepare MoE model with LoRA for memory-optimized full precision fine-tuning."""
    # Clear any cached memory first
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    
    print("Loading tokenizer...")
    tokenizer = AutoTokenizer.from_pretrained(
        config.model_name,
        trust_remote_code=True,
        padding_side="right",
    )
    
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    
    dtype = torch.bfloat16 if config.bf16 else (torch.float16 if config.fp16 else torch.float32)
    
    print(f"\nLoading MoE base model in {dtype} precision...")
    print("This may take several minutes for a 20b MoE model...")
    
    # CRITICAL: For MoE models, avoid CPU offloading which causes the KeyError
    # Load everything directly to GPU
    model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        trust_remote_code=True,
        torch_dtype=dtype,
        device_map="auto",  # Let it handle MoE architecture automatically
        low_cpu_mem_usage=True,
        attn_implementation="flash_attention_2",
        # Don't use offload_folder or max_memory for MoE models
    )
    
    # CRITICAL: Disable cache for training
    model.config.use_cache = False
    
    print("\nConfiguring LoRA for MoE model...")
    print("Note: Targeting only attention layers to avoid MoE expert complications")
    
    lora_config = LoraConfig(
        r=config.lora_r,
        lora_alpha=config.lora_alpha,
        target_modules=config.lora_target_modules,
        lora_dropout=config.lora_dropout,
        bias=config.lora_bias,
        task_type=TaskType.CAUSAL_LM,
        modules_to_save=None,
    )
    
    print("Applying LoRA adapters...")
    model = get_peft_model(model, lora_config)
    
    # CRITICAL: Enable gradient checkpointing AFTER applying LoRA
    if config.gradient_checkpointing:
        print("Enabling gradient checkpointing (use_reentrant=False)...")
        model.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs={"use_reentrant": config.gradient_checkpointing_use_reentrant}
        )
        
        # Enable input to require gradients
        if hasattr(model, 'enable_input_require_grads'):
            model.enable_input_require_grads()
            print("Enabled input_require_grads")
        else:
            # Fallback for older versions
            def make_inputs_require_grad(module, input, output):
                output.requires_grad_(True)
            model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)
            print("Enabled input_require_grads (fallback method)")
    
    # Verify trainable parameters
    model.print_trainable_parameters()
    
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    # Verify LoRA parameters
    lora_params_count = 0
    for name, param in model.named_parameters():
        if 'lora_' in name and param.requires_grad:
            lora_params_count += 1
    
    # Check current memory usage
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated(0) / 1024**3
        reserved = torch.cuda.memory_reserved(0) / 1024**3
        
        print(f"\n{'='*80}")
        print(f"MEMORY STATUS AFTER MODEL LOADING")
        print(f"{'='*80}")
        print(f"GPU Memory Allocated: {allocated:.2f}GB")
        print(f"GPU Memory Reserved: {reserved:.2f}GB")
        print(f"{'='*80}\n")
    
    print(f"\n{'='*80}")
    print(f"MODEL CONFIGURATION (MoE)")
    print(f"{'='*80}")
    print(f"Model: {config.model_name}")
    print(f"Architecture: Mixture of Experts (MoE)")
    print(f"Precision: {dtype}")
    print(f"Attention: Flash Attention 2")
    print(f"\nLoRA Configuration:")
    print(f"  - Rank (r): {config.lora_r}")
    print(f"  - Alpha: {config.lora_alpha}")
    print(f"  - Dropout: {config.lora_dropout}")
    print(f"  - Target modules: {', '.join(config.lora_target_modules)}")
    print(f"  - Note: Only attention layers (avoiding MoE experts)")
    print(f"\nParameter Statistics:")
    print(f"  - Total parameters: {total_params:,}")
    print(f"  - Trainable parameters: {trainable_params:,}")
    print(f"  - Trainable %: {100 * trainable_params / total_params:.4f}%")
    print(f"  - LoRA parameters with grad: {lora_params_count}")
    print(f"\nOptimizations:")
    print(f"  - Gradient checkpointing: {'Enabled' if config.gradient_checkpointing else 'Disabled'}")
    print(f"  - Use reentrant: {config.gradient_checkpointing_use_reentrant}")
    print(f"  - CPU offloading: Disabled (MoE compatibility)")
    print(f"{'='*80}\n")
    
    # Final verification
    if trainable_params == 0:
        raise RuntimeError("No trainable parameters found! Check LoRA configuration.")
    
    print("MoE model ready for H100 training\n")
    
    return model, tokenizer

# ============================================================================
# Custom Callbacks
# ============================================================================

class MetricsCallback(TrainerCallback):
    """Callback to log metrics clearly."""
    
    def __init__(self):
        self.train_losses = []
        self.eval_losses = []
        self.steps = []
        self.eval_steps = []
    
    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs is not None:
            if "loss" in logs:
                self.train_losses.append(logs['loss'])
                self.steps.append(state.global_step)
                
                print(f"\n{'='*80}")
                print(f"Step: {state.global_step}/{state.max_steps}")
                print(f"Epoch: {state.epoch:.2f}/{args.num_train_epochs}")
                print(f"Training Loss: {logs['loss']:.4f}")
                
                if "learning_rate" in logs:
                    print(f"Learning Rate: {logs['learning_rate']:.2e}")
                    
            if "eval_loss" in logs:
                self.eval_losses.append(logs['eval_loss'])
                self.eval_steps.append(state.global_step)
                
                print(f"\n{'='*80}")
                print(f"EVALUATION - Step: {state.global_step}")
                print(f"Validation Loss: {logs['eval_loss']:.4f}")
                
                if len(self.eval_losses) > 1:
                    prev_loss = self.eval_losses[-2]
                    improvement = prev_loss - logs['eval_loss']
                    print(f"Change from previous: {improvement:+.4f}")
                    
                if "eval_runtime" in logs:
                    print(f"Eval Runtime: {logs['eval_runtime']:.2f}s")
                if "eval_samples_per_second" in logs:
                    print(f"Eval Samples/sec: {logs['eval_samples_per_second']:.2f}")
                    
                print(f"{'='*80}\n")
    
    def on_train_end(self, args, state, control, **kwargs):
        """Print summary statistics at the end of training."""
        if self.train_losses:
            print(f"\n{'='*80}")
            print("TRAINING SUMMARY")
            print(f"{'='*80}")
            print(f"Initial Training Loss: {self.train_losses[0]:.4f}")
            print(f"Final Training Loss: {self.train_losses[-1]:.4f}")
            print(f"Training Loss Reduction: {self.train_losses[0] - self.train_losses[-1]:.4f}")
            
        if self.eval_losses:
            print(f"\nInitial Validation Loss: {self.eval_losses[0]:.4f}")
            print(f"Final Validation Loss: {self.eval_losses[-1]:.4f}")
            print(f"Validation Loss Reduction: {self.eval_losses[0] - self.eval_losses[-1]:.4f}")
            print(f"Best Validation Loss: {min(self.eval_losses):.4f}")
            print(f"{'='*80}\n")

# ============================================================================
# Training
# ============================================================================

def train(config: Config):
    """Main training function with memory optimization for MoE models."""
    
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"\n{'='*80}")
        print(f"H100 GPU CONFIGURATION")
        print(f"{'='*80}")
        props = torch.cuda.get_device_properties(0)
        print(f"GPU: {props.name}")
        print(f"Total Memory: {props.total_memory / 1024**3:.2f} GB")
        print(f"Compute Capability: {props.major}.{props.minor}")
        print(f"Multi-Processor Count: {props.multi_processor_count}")
        print(f"{'='*80}\n")
    
    print("\nLoading MoE model and tokenizer with memory-optimized LoRA...")
    model, tokenizer = setup_model_and_tokenizer(config)
    
    print("Loading and preparing datasets...")
    train_dataset, val_dataset = load_and_prepare_data(config, tokenizer)
    
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True,
        return_tensors="pt",
    )
    
    training_args = TrainingArguments(
        output_dir=config.output_dir,
        num_train_epochs=config.num_train_epochs,
        per_device_train_batch_size=config.per_device_train_batch_size,
        per_device_eval_batch_size=config.per_device_eval_batch_size,
        gradient_accumulation_steps=config.gradient_accumulation_steps,
        learning_rate=config.learning_rate,
        warmup_ratio=config.warmup_ratio,
        weight_decay=config.weight_decay,
        max_grad_norm=config.max_grad_norm,
        logging_steps=config.logging_steps,
        eval_steps=config.eval_steps,
        save_steps=config.save_steps,
        save_total_limit=config.save_total_limit,
        eval_strategy="steps",
        save_strategy="steps",
        load_best_model_at_end=False,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        bf16=config.bf16,
        fp16=config.fp16,
        bf16_full_eval=config.bf16,
        tf32=config.tf32,
        optim=config.optim,
        adam_beta1=config.adam_beta1,
        adam_beta2=config.adam_beta2,
        adam_epsilon=config.adam_epsilon,
        report_to="none",
        remove_unused_columns=False,
        gradient_checkpointing=config.gradient_checkpointing,
        gradient_checkpointing_kwargs={"use_reentrant": config.gradient_checkpointing_use_reentrant} if config.gradient_checkpointing else None,
        deepspeed=config.deepspeed,
        dataloader_pin_memory=True,
        dataloader_num_workers=2,
        dataloader_persistent_workers=False,
        eval_accumulation_steps=4,
        save_safetensors=True,
    )
    
    metrics_callback = MetricsCallback()
    memory_callback = MemoryMonitorCallback()
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        data_collator=data_collator,
        callbacks=[metrics_callback, memory_callback],
    )
    
    checkpoint = None
    if config.resume_from_checkpoint is not None:
        checkpoint = config.resume_from_checkpoint
        print(f"\n{'='*80}")
        print(f"RESUMING FROM CHECKPOINT: {checkpoint}")
        print(f"{'='*80}\n")
    
    effective_batch_size = (config.per_device_train_batch_size * 
                           config.gradient_accumulation_steps)
    
    print(f"\n{'='*80}")
    print("TRAINING CONFIGURATION (MoE)")
    print(f"{'='*80}")
    print(f"Training Mode: Memory-Optimized Full Precision LoRA")
    print(f"Architecture: Mixture of Experts (MoE)")
    print(f"GPU: H100 (Single GPU)")
    print(f"Precision: BF16" if config.bf16 else "FP32")
    print(f"Flash Attention: Enabled")
    print(f"TF32: {'Enabled' if config.tf32 else 'Disabled'}")
    print(f"\nDataset:")
    print(f"  - File: {config.dataset_file}")
    print(f"  - Train/Val split: {(1-config.val_split_ratio)*100:.0f}% / {config.val_split_ratio*100:.0f}%")
    print(f"  - Random seed: {config.random_seed}")
    print(f"  - Train samples: {len(train_dataset)}")
    print(f"  - Validation samples: {len(val_dataset)}")
    print(f"\nBatch Configuration:")
    print(f"  - Batch size per device: {config.per_device_train_batch_size}")
    print(f"  - Gradient accumulation steps: {config.gradient_accumulation_steps}")
    print(f"  - Effective batch size: {effective_batch_size}")
    print(f"\nTraining Parameters:")
    print(f"  - Number of epochs: {config.num_train_epochs}")
    print(f"  - Learning rate: {config.learning_rate}")
    print(f"  - Optimizer: {config.optim}")
    print(f"  - Adam beta1: {config.adam_beta1}")
    print(f"  - Adam beta2: {config.adam_beta2}")
    print(f"  - Warmup ratio: {config.warmup_ratio}")
    print(f"  - Weight decay: {config.weight_decay}")
    print(f"  - Max sequence length: {config.max_seq_length}")
    print(f"\nMemory Optimizations:")
    print(f"  - Gradient checkpointing: {config.gradient_checkpointing}")
    print(f"  - Use reentrant: {config.gradient_checkpointing_use_reentrant}")
    print(f"  - CPU offloading: Disabled (MoE)")
    
    total_steps = (len(train_dataset) // effective_batch_size) * config.num_train_epochs
    print(f"\nSchedule:")
    print(f"  - Estimated total steps: {total_steps}")
    print(f"  - Eval every {config.eval_steps} steps")
    print(f"  - Save every {config.save_steps} steps")
    print(f"  - Log every {config.logging_steps} steps")
    print(f"{'='*80}\n")
    
    print("Starting training...\n")
    
    try:
        trainer.train(resume_from_checkpoint=checkpoint)
    except torch.cuda.OutOfMemoryError as e:
        print(f"\n{'='*80}")
        print("OUT OF MEMORY ERROR!")
        print(f"{'='*80}")
        print(f"Error: {e}")
        print("\nSuggestions for MoE models:")
        print("1. Reduce max_seq_length to 1024")
        print("2. Reduce lora_r to 16")
        print("3. Keep only ['q_proj', 'v_proj'] for lora_target_modules")
        print("4. MoE models are memory-intensive due to multiple experts")
        print("5. Consider QLoRA (4-bit quantization) for MoE models")
        print(f"{'='*80}\n")
        raise
    except Exception as e:
        print(f"\n{'='*80}")
        print("TRAINING ERROR!")
        print(f"{'='*80}")
        print(f"Error: {e}")
        print(f"Error type: {type(e).__name__}")
        print(f"{'='*80}\n")
        raise
    
    print("\nSaving final LoRA adapters...")
    final_model_path = os.path.join(config.output_dir, "final_model")
    model.save_pretrained(final_model_path)
    tokenizer.save_pretrained(final_model_path)
    print(f"Final LoRA adapters saved to: {final_model_path}")
    print("Note: To use the model, load the base model and then load these adapters.")
    
    print("\n✓ Training completed successfully!")
    
    print("\nRunning final evaluation...")
    eval_results = trainer.evaluate()
    print(f"\n{'='*80}")
    print("FINAL EVALUATION RESULTS")
    print(f"{'='*80}")
    for key, value in eval_results.items():
        if isinstance(value, float):
            print(f"{key}: {value:.4f}")
        else:
            print(f"{key}: {value}")
    print(f"{'='*80}\n")
    
    # Final memory report
    if torch.cuda.is_available():
        print(f"{'='*80}")
        print("FINAL MEMORY REPORT")
        print(f"{'='*80}")
        print(f"Peak Memory Allocated: {torch.cuda.max_memory_allocated(0) / 1024**3:.2f}GB")
        print(f"Peak Memory Reserved: {torch.cuda.max_memory_reserved(0) / 1024**3:.2f}GB")
        print(f"{'='*80}\n")

# ============================================================================
# Main with Error Handling
# ============================================================================

def main():
    """Main entry point optimized for MoE models."""
    
    # Set environment variable to avoid tokenizer warnings
    os.environ["TOKENIZERS_PARALLELISM"] = "false"
    
    config = Config(
        # Dataset
        dataset_file="/kaggle/input/datasets/jorgeplazas/numinamath-tir/NuminaMath-TIR.csv",
        val_split_ratio=0.05,
        random_seed=42,
        
        # Conservative LoRA settings for 20b MoE model
        lora_r=64,
        lora_alpha=128,
        lora_target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],  # Attention only for MoE
        
        # Memory-optimized training settings
        output_dir="./lora_outputs",
        num_train_epochs=2,
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        max_seq_length=8192,  # Conservative for MoE
        
        # Evaluation
        eval_steps=250,
        save_steps=250,
        logging_steps=10,
        save_total_limit=2,
        
        # H100 optimizations
        gradient_checkpointing=True,
        gradient_checkpointing_use_reentrant=False,
        bf16=True,
        tf32=True,
        optim="adamw_torch_fused",
        adam_beta1=0.9,
        adam_beta2=0.95,
        
        # MoE specific
        use_cpu_offload=False,  # Critical: disabled for MoE
        
        resume_from_checkpoint='',
    )
    
    print(f"\n{'='*80}")
    print("STARTING TRAINING - MoE MODEL OPTIMIZED")
    print(f"{'='*80}")
    print("MoE-specific optimizations:")
    print("  - CPU offloading disabled (causes KeyError with experts)")
    print("  - Only attention layers targeted with LoRA")
    print("  - Conservative memory settings")
    print("\nIf training succeeds, you can gradually:")
    print("  - Increase max_seq_length (2048 -> 4096)")
    print("  - Increase lora_r (32 -> 64)")
    print("  - Note: Adding MLP/expert layers may cause issues")
    print(f"{'='*80}\n")
    
    try:
        train(config)
    except Exception as e:
        print(f"\n{'='*80}")
        print("TRAINING FAILED")
        print(f"{'='*80}")
        print(f"Error: {e}")
        print(f"Error type: {type(e).__name__}")
        print("\nFor MoE models, if you still have issues:")
        print("1. This model architecture may require specific handling")
        print("2. Consider using smaller sequence lengths (1024)")
        print("3. Reduce LoRA rank to 16")
        print("4. Try QLoRA (4-bit quantization) for better memory efficiency")
        print(f"{'='*80}\n")
        raise

if __name__ == "__main__":
    main()